In [37]:
!pip install sklearn-crfsuite "scikit-learn<0.24" zeyrek

In [38]:
from collections import Counter

import sklearn_crfsuite
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import scorers
from sklearn_crfsuite import metrics
import numpy as np
from tabulate import tabulate
import zeyrek
import nltk

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

**Create training data**

In [39]:
with open('/content/drive/MyDrive/NLP2/train.txt', 'r', encoding="utf-8-sig") as fd:
    train = fd.readlines()

In [40]:
allowed_chars = "abcçdefgğhıijklmnoöprsştuüvyz0123456789`'"

# Used to translate critical characters to uppercase ASCII
diacritize_map = {
    ord(u'c'): u'C',
    ord(u'ç'): u'C',
    ord(u'g'): u'G',
    ord(u'ğ'): u'G',
    ord(u'ı'): u'I',
    ord(u'i'): u'I',
    ord(u'o'): u'O',
    ord(u'ö'): u'O',
    ord(u'u'): u'U',
    ord(u'ü'): u'U',
    ord(u'ş'): u'S',
    ord(u's'): u'S'
}
# Used to check if a word contains critical characters or not
diacritic_letters = "cçgğıioöuüsş"
# Map for creating class labels for Turkish characters
turkish_map = {
    "ç": "ch",
    "ğ": "soft_g",
    "ı": "lower_I",
    "ö": "oe",
    "ü": "ue",
    "ş": "sh"
}


def contains_only_allowed_chars(word):
    """
    This function checks to see if a given
    word contains only the allowed chars
    or no.
    """
    for c in word:
        if c not in allowed_chars:
            return False
    return True


def convert_to_diaform(word):
    """
    Converts critical characters to uppercase
    ASCII form. olduğunu -> OldUGUnU
    """
    return word.translate(diacritize_map)


def sentence_to_train_data(sentence):
    """
    Extracts features from a given sentence
    and creates training data
    """
    X = []
    y = []
    for ix, word in enumerate(sentence):
        word_diaform = convert_to_diaform(word)
        for iy, letter in enumerate(word):
            if letter in diacritic_letters:
                features = {}
                if ix > 0:
                    features["prev_word"] = convert_to_diaform(sentence[ix - 1])
                if iy > 4:
                    features["ch_-5"] = word_diaform[iy - 5]
                if iy > 3:
                    features["ch_-4"] = word_diaform[iy - 4]
                if iy > 2:
                    features["ch_-3"] = word_diaform[iy - 3]
                if iy > 1:
                    features["ch_-2"] = word_diaform[iy - 2]
                if iy > 0:
                    features["ch_-1"] = word_diaform[iy - 1]
                if iy < len(word) - 5:
                    features["ch_+5"] = word_diaform[iy + 5]
                if iy < len(word) - 4:
                    features["ch_+4"] = word_diaform[iy + 4]
                if iy < len(word) - 3:
                    features["ch_+3"] = word_diaform[iy + 3]
                if iy < len(word) - 2:
                    features["ch_+2"] = word_diaform[iy + 2]
                if iy < len(word) - 1:
                    features["ch_+1"] = word_diaform[iy + 1]
                features["cur_letter"] = word_diaform[iy]
                features["cur_word"] = word_diaform
                if len(word_diaform) >= 3:
                    features["cur_word_3"] = word_diaform[:3]
                if len(word_diaform) >= 5:
                    features["cur_word_5"] = word_diaform[:5]
                if len(word_diaform) >= 7:
                    features["cur_word_7"] = word_diaform[:7]
                if "a" in word_diaform:
                    features["contains_a"] = "1"
                else:
                    features["contains_a"] = "0"
                if "e" in word_diaform:
                    features["contains_e"] = "1"
                else:
                    features["contains_e"] = "0"
                X.append(features)
                y.append(f"[{letter if letter not in turkish_map.keys() else turkish_map[letter]}]")
    return X, y


def extract_training_data(sentence):
    """
    Creates training data from a given sentence
    """
    sentence_words = np.array(sentence.split(" "))
    allowed_words_map = [contains_only_allowed_chars(word) and len(word) <= 20 for word in sentence_words]
    allowed_words = sentence_words[allowed_words_map]
    return sentence_to_train_data(allowed_words)

In [41]:
X_all = []
y_all = []

# Extract training data from all sentencess
for sentence in train:
    X, y = extract_training_data(sentence)
    X_all.append(X)
    y_all.append(y)

# Split training data as train and test
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.25, random_state=42)

In [42]:
def print_train_table():
    """
    Prints the first row of the training data.
    """
    columns = ["cur_letter", "cur_word", "prev_word", "next_word", 
               "ch_-2", "ch_-1", 
               "ch_+1", "ch_+2"]
    headers = columns + ["class"]
    data = []
    for ix, x in enumerate(X_train[0]):
        xd = []
        for column in columns:
            if column in x:
                xd.append(x[column])
            else:
                xd.append("-")
        xd.append(y_train[0][ix])
        data.append(xd)
    print(tabulate(data, headers=headers))

print("NOTE: ch_-4, ch_-3, ch_+3, and ch_+4 columns are not shown in \n\
the table because it was too wide, however those information is in \nthe training data.")
print("")
print_train_table()

NOTE: ch_-4, ch_-3, ch_+3, and ch_+4 columns are not shown in 
the table because it was too wide, however those information is in 
the training data.

cur_letter    cur_word    prev_word    next_word    ch_-2    ch_-1    ch_+1    ch_+2    class
------------  ----------  -----------  -----------  -------  -------  -------  -------  ---------
G             GeCen       her          -            -        -        e        C        [g]
C             GeCen       her          -            G        e        e        n        [ch]
G             GUn         GeCen        -            -        -        U        n        [g]
U             GUn         GeCen        -            -        G        n        -        [ue]
O             Ona         GUn          -            -        -        n        a        [o]
I             bIraz       Ona          -            -        b        r        a        [i]
S             aSIk        fazla        -            -        a        I        k        [sh]
I         

**Train the model**

In [43]:
%%time

# Create and train the model
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs', 
    c1=0.1, 
    c2=0.001, 
    max_iterations=100, 
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

CPU times: user 2min 22s, sys: 349 ms, total: 2min 22s
Wall time: 2min 22s


**Evaluate the model**

In [44]:
labels = list(crf.classes_)
y_pred = crf.predict(X_test)
f1 = metrics.flat_f1_score(y_test, y_pred, 
                      average='weighted', labels=labels)
print(f"F1-Score on test split: {f1:.4f}%")

F1-Score on test split: 0.9854%


In [45]:
# Print precision, recall, and f1-score for all classes
sorted_labels = sorted(
    labels, 
    key=lambda name: (name[1:], name[0])
)
print(metrics.flat_classification_report(
    y_test, y_pred, labels=sorted_labels, digits=3
))

              precision    recall  f1-score   support

         [c]      0.987     0.979     0.983     10219
        [ch]      0.981     0.988     0.985     11279
         [g]      0.997     0.989     0.993     13496
         [i]      0.989     0.989     0.989     90270
   [lower_I]      0.980     0.979     0.980     50093
         [o]      0.992     0.994     0.993     27138
        [oe]      0.982     0.972     0.977      8386
         [s]      0.982     0.988     0.985     34295
        [sh]      0.975     0.963     0.969     17082
    [soft_g]      0.986     0.996     0.991     10859
         [u]      0.988     0.990     0.989     32784
        [ue]      0.983     0.979     0.981     19546

    accuracy                          0.985    325447
   macro avg      0.985     0.984     0.984    325447
weighted avg      0.985     0.985     0.985    325447



In [46]:
# This cells prints hte most likely and unlikely
# state changes in the model

def print_transitions(trans_features):
    for (label_from, label_to), weight in trans_features:
        print("%-6s -> %-7s %0.6f" % (label_from, label_to, weight))

print("Top likely transitions:")
print_transitions(Counter(crf.transition_features_).most_common(10))

print("\nTop unlikely transitions:")
print_transitions(reversed(Counter(crf.transition_features_).most_common()[-10:]))

Top likely transitions:
[oe]   -> [ue]    3.365927
[ch]   -> [lower_I] 2.251883
[ue]   -> [ue]    2.170617
[oe]   -> [soft_g] 1.281637
[ch]   -> [oe]    1.264085
[oe]   -> [ch]    1.237981
[soft_g] -> [lower_I] 1.193888
[ue]   -> [soft_g] 1.115478
[o]    -> [u]     1.033402
[g]    -> [ue]    0.976057

Top unlikely transitions:
[oe]   -> [u]     -2.849778
[c]    -> [soft_g] -1.757399
[soft_g] -> [o]     -1.459023
[soft_g] -> [g]     -1.373417
[soft_g] -> [s]     -1.136829
[g]    -> [soft_g] -1.110702
[oe]   -> [i]     -1.088230
[c]    -> [ue]    -1.053232
[g]    -> [c]     -1.016552
[soft_g] -> [soft_g] -0.986732


In [47]:
# This cell prints the top positive and negative 
# feature for a class to be activated. As you can
# see cur_letter:G is the most positive indicator
# that the class label will be G.
# Also, it's very unlikely that if the cur_word
# is alI, the class label should be ı, instead
# it should always be i.

def print_state_features(state_features):
    for (attr, label), weight in state_features:
        print("%0.6f %-8s %s" % (weight, label, attr))    

print("Top positive:")
print_state_features(Counter(crf.state_features_).most_common(10))

print("\nTop negative:")
print_state_features(Counter(crf.state_features_).most_common()[-10:])

Top positive:
20.566033 [lower_I] cur_word:SOyleyebIleCekmI
20.258669 [g]      cur_letter:G
19.841385 [lower_I] cur_word:edIle
19.258515 [i]      cur_word:bIrakmamalIydI
19.115940 [s]      cur_letter:S
18.760562 [o]      cur_letter:O
18.433232 [sh]     cur_word_5:arSIv
17.776726 [c]      cur_letter:C
17.246521 [soft_g] cur_letter:G
16.887957 [sh]     cur_letter:S

Top negative:
-6.744695 [i]      prev_word:SeyrettIGImIz
-6.794587 [ch]     cur_word_3:Cla
-6.798962 [o]      cur_word_3:SUl
-6.820308 [c]      ch_-1:t
-7.261468 [i]      cur_word:GenIS
-7.478579 [i]      cur_word:GeCmIStekI
-7.685152 [ch]     ch_-1:m
-7.812252 [i]      cur_word:OGretmenI
-7.815987 [c]      ch_-1:f
-9.556037 [sh]     cur_word_3:lIS


**Convert text to restore Diacritics**

In [48]:
diacritic_uppercases = "CGIOUS"

# Will be used to convert from model output to Turkish
dia_map = {
    "[c]": "c",
    "[ch]": "ç",
    "[g]": "g",
    "[soft_g]": "ğ",
    "[i]": "i",
    "[lower_I]": "ı",
    "[o]": "o",
    "[oe]": "ö",
    "[u]": "u",
    "[ue]": "ü",
    "[s]": "s",
    "[sh]": "ş"
}


def convert_to_result(sentence, pred):
    """
    This function iterates over a list of words (sentence)
    and converts each word to Turkish.
    """
    index = 0
    words = []
    for word in sentence:
        res_word = ""
        for letter in word:
            if letter in diacritic_uppercases:
                res_word += dia_map[pred[0][index]]
                index += 1
            else:
                res_word += letter
        words.append(res_word)
    return words


def diacritic_restore(sentence, crf):
    """
    This function, given an ASCII sentence, does the 
    diacritic restoration operation. 
    """
    sentence_words = np.array(sentence.split(" ")) # tokenize sentence
    allowed_words_map = np.array([contains_only_allowed_chars(word) for word in sentence_words])
    # This is the words the model will have predictions on
    allowed_words = sentence_words[allowed_words_map]
    # Convert sentence to the train data form
    X, _ = sentence_to_train_data(allowed_words)
    pred = crf.predict([X])  # Do the prediction
    # Converts all words to diacritic form to be processed later
    allowed_words_diaform = [convert_to_diaform(x) for x in allowed_words]
    # Converts diacritic words to results
    diacritized_words = np.array(convert_to_result(allowed_words_diaform, pred))
    # Put resulting words to correct places in sentences
    np.put(sentence_words, np.argwhere(allowed_words_map==True).flatten(), diacritized_words)
    return " ".join(sentence_words)

In [49]:
diacritic_restore("gulsen cebiroglu eryigit", crf)

'gülşen cebiroğlu eryiğit'

**Test on test data**

In [57]:
# read test set (X)
with open('/content/drive/MyDrive/NLP2/test_new.txt', 'r') as fd:
    test = fd.readlines()

In [58]:
# Restore diacritics in test data
test = [diacritic_restore(sentence, crf) for sentence in test]

In [59]:
# read labeled(diacritized) test set (Y)
with open('/content/drive/MyDrive/NLP2/testgold_new.txt', 'r', encoding="utf-8") as fd:
    testgold = fd.readlines()

In [60]:
# accuracy over the entire words 
def acc_overall(test_result, testgold):
  
  correct = 0
  total = 0
  # count number of correctly diacritized words
  for i in range(len(testgold)):
    for m in range(len(testgold[i].split())):
      if test_result[i].split()[m] == testgold[i].split()[m]:
        correct += 1
      total +=1

  return correct / total

In [61]:
# read ambigious words
with open('/content/drive/MyDrive/NLP2/belirsizler.txt', 'r', encoding="utf-8") as fd:
    amb = fd.readlines()

In [62]:
# accuracy over the ambigious words alone 
def acc_amb(test_result, testgold):
  
  analyzer = zeyrek.MorphAnalyzer()
  
  correct = 0
  total = 0
  # count number of correctly diacritized ambigious words
  for i in range(len(testgold)):
    for m in range(len(testgold[i].split())):
      if analyzer.lemmatize(testgold[i].split()[m])[0][1][0] + "\n" in amb:
        if test_result[i].split()[m] == testgold[i].split()[m]:
          correct += 1
        total +=1

  return correct / total

In [64]:
print(acc_overall(test, testgold))

IndexError: ignored